# P2(a) 후보 쌍 v1 코사인 (Colab GPU)

`docs/SERVICE_PIPELINE_PLAN.md` 11-21. 평가 run에 임베딩 상위 50개만 있어 문자 BM25·LLM 쿼리 확장으로만 들어온 후보의 v1 점수가 0으로 처리됐다.
서비스처럼 모든 후보에 실제 코사인을 주려고, 점수가 없는 (쿼리, 가맹점) 쌍만 v1으로 채점한다.

1. 로컬: `python scripts/export_p2_cosine.py` → `colab_upload/p2_cosine/`(pairs·queries·docs·meta)를 Drive `내 드라이브/store-search-ai/p2_cosine/`에 올림
2. Colab: 런타임 GPU(T4 가능) → 위에서부터 실행 → 같은 폴더에 `pair_cosine_<모델>.csv`(query_id, doc_id, cosine)
3. 로컬: 그 파일을 `results/service_pipeline/p2/`에 넣음

인코딩 조건은 `colab/service_dense_runs.ipynb`와 같다(fp16, 정규화, 쿼리 지시문 없음, 문서 = search_text_t1_minimal).

In [ ]:
import torch
assert torch.cuda.is_available(), "런타임 > 런타임 유형 변경 > GPU를 선택하세요"
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DIR = Path("/content/drive/MyDrive/store-search-ai/p2_cosine")
NAME = "bge_m3_ft_mj_20261014_0743"
MODEL = "/content/drive/MyDrive/store-search-ai/runs/finetune/bge_m3_ft_mj_20261014_0743"
BATCH = 256

pairs = pd.read_csv(DIR / "pairs.csv", encoding="utf-8-sig", dtype=str)
queries = pd.read_csv(DIR / "queries.csv", encoding="utf-8-sig", dtype=str)
docs = pd.read_csv(DIR / "docs.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
print(len(pairs), "쌍,", len(queries), "쿼리,", len(docs), "가맹점")

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL, device="cuda", model_kwargs={"torch_dtype": torch.float16})
q_emb = model.encode(queries["query"].tolist(), batch_size=BATCH, normalize_embeddings=True, convert_to_numpy=True).astype("float32")
d_emb = model.encode(docs["text"].tolist(), batch_size=BATCH, normalize_embeddings=True, show_progress_bar=True,
                     convert_to_numpy=True).astype("float32")
qi = pd.Series(np.arange(len(queries)), index=queries["query_id"])
di = pd.Series(np.arange(len(docs)), index=docs["doc_id"])
a, b = qi.loc[pairs["query_id"]].to_numpy(), di.loc[pairs["doc_id"]].to_numpy()
cos = np.einsum("ij,ij->i", q_emb[a], d_emb[b])
out = DIR / f"pair_cosine_{NAME}.csv"
pairs.assign(cosine=cos).to_csv(out, index=False, encoding="utf-8-sig", lineterminator="
")
print("[저장]", out, pd.Series(cos).describe().round(3).to_dict())